In [1]:
! pip install dgeb esm torch scikit-learn matplotlib datasets seaborn rdflib einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 196.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2

In [2]:
# from rdflib import Graph

# file_path = "/content/known_operon.ttl"

# g = Graph()
# g.parse(file_path, format="turtle")

# print("Number of triples:", len(g))
# print()

# # Print first 20 triples
# for i, triple in enumerate(g):
#     print(triple)
#     if i >= 20:
#         break

In [2]:
import argparse
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    f1_score,
    accuracy_score,
)
from sklearn.preprocessing import StandardScaler




In [3]:
import einops

In [4]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [5]:
# ── Cell 2: config — must run before anything else ───────────────────────────
import os

# define DRIVE_DIR first — everything else depends on this
DRIVE_DIR       = "/content/drive/MyDrive/dgeb_siamese"
OPERON_DATA_DIR = f"{DRIVE_DIR}/operon_data"

# exact column names from your CSV
COL_SEQ1   = "sequence_1"
COL_SEQ2   = "sequence_2"
COL_OPERON = "operon_id"
COL_LABEL  = "label"

# file paths — both defined here, before the loader function
TRAIN_CSV = f"{OPERON_DATA_DIR}/operon_train.csv"
VAL_CSV   = f"{OPERON_DATA_DIR}/operon_val.csv"

# create folders if they don't exist yet
os.makedirs(OPERON_DATA_DIR, exist_ok=True)

# confirm files are visible
print(f"DRIVE_DIR      : {DRIVE_DIR}")
print(f"OPERON_DATA_DIR: {OPERON_DATA_DIR}")
print()
for path, name in [(TRAIN_CSV, "operon_train.csv"), (VAL_CSV, "operon_val.csv")]:
    exists = os.path.exists(path)
    print(f"  {name} : {'FOUND' if exists else 'NOT FOUND — check your Drive path'}")

DRIVE_DIR      : /content/drive/MyDrive/dgeb_siamese
OPERON_DATA_DIR: /content/drive/MyDrive/dgeb_siamese/operon_data

  operon_train.csv : FOUND
  operon_val.csv : FOUND


In [6]:

import os
import pandas as pd

def load_operon_csv(split: str) -> pd.DataFrame:
    """
    Load operon_train.csv or operon_val.csv from Drive.

    Parameters
    ----------
    split : "train" or "val"

    Returns
    -------
    pd.DataFrame with columns: sequence_1, sequence_2, operon_id, label
    """
    path = TRAIN_CSV if split == "train" else VAL_CSV

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nFile not found: {path}\n"
            f"Make sure you uploaded operon_train.csv and operon_val.csv "
            f"to {OPERON_DATA_DIR}"
        )

    df = pd.read_csv(path)

    # ── validate columns ──────────────────────────────────────────────────
    required = {COL_SEQ1, COL_SEQ2, COL_OPERON, COL_LABEL}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing columns in {os.path.basename(path)}: {missing}\n"
            f"Found: {list(df.columns)}"
        )

    # ── drop nulls ────────────────────────────────────────────────────────
    before = len(df)
    df = df.dropna(subset=[COL_SEQ1, COL_SEQ2, COL_LABEL])
    if before - len(df):
        print(f"  Dropped {before - len(df)} rows with null values.")

    # ── coerce types ──────────────────────────────────────────────────────
    df[COL_SEQ1]   = df[COL_SEQ1].astype(str).str.strip()
    df[COL_SEQ2]   = df[COL_SEQ2].astype(str).str.strip()
    df[COL_LABEL]  = df[COL_LABEL].astype(int)
    df[COL_OPERON] = df[COL_OPERON].astype(str).str.strip()

    return df


def df_to_records(df: pd.DataFrame) -> list:
    """
    Convert a DataFrame to a list of dicts expected by the embedding functions.
    """
    return [
        {
            "seq1":     row[COL_SEQ1],
            "seq2":     row[COL_SEQ2],
            "operon_id": row[COL_OPERON],
            "label":    row[COL_LABEL],
        }
        for _, row in df.iterrows()
    ]


# ── load both splits ──────────────────────────────────────────────────────────
print("Loading operon_train.csv ...")
train_df = load_operon_csv("train")
print(f"  rows     : {len(train_df)}")
print(f"  positive : {train_df[COL_LABEL].sum()} ({100*train_df[COL_LABEL].mean():.1f}%)")
print(f"  negative : {(train_df[COL_LABEL]==0).sum()}")
print(f"  operons  : {train_df[COL_OPERON].nunique()} unique operon IDs")

print()
print("Loading operon_val.csv ...")
val_df = load_operon_csv("val")
print(f"  rows     : {len(val_df)}")
print(f"  positive : {val_df[COL_LABEL].sum()} ({100*val_df[COL_LABEL].mean():.1f}%)")
print(f"  negative : {(val_df[COL_LABEL]==0).sum()}")
print(f"  operons  : {val_df[COL_OPERON].nunique()} unique operon IDs")

# convert to records for the embedding pipeline
train_records = df_to_records(train_df)
val_records   = df_to_records(val_df)

Loading operon_train.csv ...
  rows     : 37126
  positive : 18598 (50.1%)
  negative : 18528
  operons  : 4886 unique operon IDs

Loading operon_val.csv ...
  rows     : 15912
  positive : 7946 (49.9%)
  negative : 7966
  operons  : 3081 unique operon IDs


In [7]:
## logistic regression

# ── Cell D: Logistic Regression with sklearn ──────────────────────────────────

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix
)

# ── 1. Feature engineering ────────────────────────────────────────────────────

def records_to_features(records: list) -> tuple[np.ndarray, np.ndarray]:
    """
    Convert sequence-pair records into hand-crafted features.

    Current features (all derived from raw sequence strings):
        0  len_seq1            – length of sequence 1
        1  len_seq2            – length of sequence 2
        2  len_diff            – |len1 - len2|
        3  len_ratio           – min/max length (0–1)
        4  gc_seq1             – GC content of seq1
        5  gc_seq2             – GC content of seq2
        6  gc_diff             – |gc1 - gc2|
        7  shared_kmers_3      – Jaccard similarity on 3-mers
        8  shared_kmers_4      – Jaccard similarity on 4-mers
        9  shared_kmers_6      – Jaccard similarity on 6-mers

    Returns
    -------
    X : np.ndarray, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    """

    def gc_content(seq: str) -> float:
        seq = seq.upper()
        gc  = seq.count("G") + seq.count("C")
        return gc / len(seq) if len(seq) > 0 else 0.0

    def kmer_jaccard(s1: str, s2: str, k: int) -> float:
        set1 = {s1[i:i+k] for i in range(len(s1) - k + 1)}
        set2 = {s2[i:i+k] for i in range(len(s2) - k + 1)}
        if not set1 and not set2:
            return 0.0
        return len(set1 & set2) / len(set1 | set2)

    # ── NEW: nucleotide frequency vector ──────────────────────────────────
    def nuc_freq(seq: str) -> dict:
        seq = seq.upper()
        n   = len(seq) if len(seq) > 0 else 1
        return {b: seq.count(b) / n for b in "ACGT"}

    # ── NEW: dinucleotide frequency vector ────────────────────────────────
    def dinuc_freq(seq: str) -> dict:
        seq    = seq.upper()
        dinucs = [a+b for a in "ACGT" for b in "ACGT"]   # 16 combos
        n      = len(seq) - 1 if len(seq) > 1 else 1
        return {d: seq.count(d) / n for d in dinucs}

    # ── NEW: kmer frequency cosine similarity ─────────────────────────────
    def kmer_cosine(s1: str, s2: str, k: int) -> float:
        def freq(s):
            n = len(s) - k + 1
            counts = {}
            for i in range(n):
                km = s[i:i+k]
                counts[km] = counts.get(km, 0) + 1
            total = sum(counts.values()) or 1
            return {km: c/total for km, c in counts.items()}
        f1, f2  = freq(s1), freq(s2)
        keys    = set(f1) | set(f2)
        dot     = sum(f1.get(k,0) * f2.get(k,0) for k in keys)
        norm1   = sum(v**2 for v in f1.values()) ** 0.5
        norm2   = sum(v**2 for v in f2.values()) ** 0.5
        return dot / (norm1 * norm2) if norm1 * norm2 > 0 else 0.0

    # ── NEW: longest common subsequence ratio ─────────────────────────────
    def lcs_ratio(s1: str, s2: str) -> float:
        # Use short sequences to keep it fast; truncate to 200 chars
        s1, s2 = s1[:200], s2[:200]
        m, n   = len(s1), len(s2)
        dp     = [[0]*(n+1) for _ in range(m+1)]
        for i in range(1, m+1):
            for j in range(1, n+1):
                dp[i][j] = dp[i-1][j-1] + 1 if s1[i-1] == s2[j-1] else max(dp[i-1][j], dp[i][j-1])
        return dp[m][n] / max(m, n) if max(m, n) > 0 else 0.0

    # ── NEW: AT content ───────────────────────────────────────────────────
    def at_content(seq: str) -> float:
        seq = seq.upper()
        return (seq.count("A") + seq.count("T")) / len(seq) if len(seq) > 0 else 0.0


    X, y = [], []
    for rec in records:
        s1, s2 = rec["seq1"], rec["seq2"]

        l1, l2   = len(s1), len(s2)
        gc1, gc2 = gc_content(s1), gc_content(s2)
        at1, at2 = at_content(s1), at_content(s2)
        nf1, nf2 = nuc_freq(s1), nuc_freq(s2)
        df1, df2 = dinuc_freq(s1), dinuc_freq(s2)


        feats = [
            l1,
            l2,
            abs(l1 - l2),
            min(l1, l2) / max(l1, l2) if max(l1, l2) > 0 else 0.0,
            gc1,
            gc2,
            abs(gc1 - gc2),
            kmer_jaccard(s1, s2, 3),
            kmer_jaccard(s1, s2, 4),
            kmer_jaccard(s1, s2, 6),
            at1, at2, abs(at1 - at2),

        ]

        # ── NEW: per-nucleotide frequencies for each seq (13–20) ──────────
        for b in "ACGT":
            feats.append(nf1[b])
        for b in "ACGT":
            feats.append(nf2[b])

        # ── NEW: per-nucleotide frequency differences (21–24) ─────────────
        for b in "ACGT":
            feats.append(abs(nf1[b] - nf2[b]))

        # ── NEW: dinucleotide frequency differences (25–40) ───────────────
        for d in [a+b for a in "ACGT" for b in "ACGT"]:
            feats.append(abs(df1[d] - df2[d]))

        # ── NEW: kmer cosine similarity (41–43) ───────────────────────────
        feats += [
            kmer_cosine(s1, s2, 3),
            kmer_cosine(s1, s2, 4),
            kmer_cosine(s1, s2, 6),
        ]

        # ── NEW: LCS ratio (44) ───────────────────────────────────────────
        feats.append(lcs_ratio(s1, s2))


        X.append(feats)
        y.append(rec["label"])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


print("Building features ...")
X_train, y_train = records_to_features(train_records)
print(X_train.shape)
print(y_train.shape)
X_val,   y_val   = records_to_features(val_records)
print(f"  Train : {X_train.shape}  |  Val : {X_val.shape}")

# ── 2. Scale ──────────────────────────────────────────────────────────────────

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit on train only
X_val   = scaler.transform(X_val)         # apply same scale to val

# ── 3. Train ──────────────────────────────────────────────────────────────────

clf = LogisticRegression(
    C             = 1.0,        # inverse regularisation strength
    max_iter      = 1_000,
    class_weight  = "balanced", # handles label imbalance automatically
    solver        = "lbfgs",
    random_state  = 42,
)

print("\nFitting logistic regression ...")
clf.fit(X_train, y_train)
print("  Done.")

# ── 4. Evaluate ───────────────────────────────────────────────────────────────

y_pred      = clf.predict(X_val)
y_prob      = clf.predict_proba(X_val)[:, 1]

acc  = accuracy_score(y_val, y_pred)
auc  = roc_auc_score(y_val, y_prob)
f1   = f1_score(y_val, y_pred)

print(f"\n{'─'*40}")
print(f"  Accuracy  : {acc:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print(f"  F1        : {f1:.4f}")
print(f"{'─'*40}")
print("\nClassification report:\n")
print(classification_report(y_val, y_pred, target_names=["negative", "positive"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_val, y_pred))

# ── 5. Feature importances ────────────────────────────────────────────────────

feature_names = [
    "len_seq1", "len_seq2", "len_diff", "len_ratio",
    "gc_seq1",  "gc_seq2",  "gc_diff",
    "jaccard_3mer", "jaccard_4mer", "jaccard_6mer",
]

coef = clf.coef_[0]
importance = sorted(zip(feature_names, coef), key=lambda x: abs(x[1]), reverse=True)

print("\nFeature coefficients (sorted by |coef|):")
print(f"  {'Feature':<18} {'Coef':>8}")
print(f"  {'─'*28}")
for name, c in importance:
    print(f"  {name:<18} {c:>+8.4f}")

Building features ...
(37126, 45)
(37126,)
  Train : (37126, 45)  |  Val : (15912, 45)

Fitting logistic regression ...
  Done.

────────────────────────────────────────
  Accuracy  : 0.5750
  ROC-AUC   : 0.6064
  F1        : 0.5761
────────────────────────────────────────

Classification report:

              precision    recall  f1-score   support

    negative       0.58      0.57      0.57      7966
    positive       0.57      0.58      0.58      7946

    accuracy                           0.57     15912
   macro avg       0.57      0.57      0.57     15912
weighted avg       0.57      0.57      0.57     15912

Confusion matrix (rows=actual, cols=predicted):
[[4554 3412]
 [3351 4595]]

Feature coefficients (sorted by |coef|):
  Feature                Coef
  ────────────────────────────
  jaccard_6mer        -0.5723
  len_seq2            -0.4127
  len_seq1            -0.3942
  jaccard_3mer        +0.3671
  jaccard_4mer        -0.3407
  len_diff            +0.3272
  len_ratio     

In [8]:
# ── Cell E: Train vs Val metrics comparison ───────────────────────────────────

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def get_metrics(clf, X, y) -> dict:
    y_pred = clf.predict(X)
    y_prob = clf.predict_proba(X)[:, 1]
    return {
        "accuracy": accuracy_score(y, y_pred),
        "roc_auc":  roc_auc_score(y, y_prob),
        "f1":       f1_score(y, y_pred),
    }

train_metrics = get_metrics(clf, X_train, y_train)
val_metrics   = get_metrics(clf, X_val,   y_val)

print(f"{'Metric':<12} {'Train':>8} {'Val':>8} {'Gap':>8}")
print("─" * 40)
for metric in ["accuracy", "roc_auc", "f1"]:
    train_val = train_metrics[metric]
    val_val   = val_metrics[metric]
    gap       = train_val - val_val
    print(f"{metric:<12} {train_val:>8.4f} {val_val:>8.4f} {gap:>+8.4f}")

Metric          Train      Val      Gap
────────────────────────────────────────
accuracy       0.5697   0.5750  -0.0053
roc_auc        0.6001   0.6064  -0.0063
f1             0.5718   0.5761  -0.0043


#Siamese Twin Architecture

## N-gram Siamese Network Architecture

```mermaid
flowchart TD
    A[Protein Sequence 1] --> B[ESM-2 Tokenizer / Encoder]
    C[Protein Sequence 2] --> D[ESM-2 Tokenizer / Encoder]

    B --> E[Sliding Window N-grams]
    D --> F[Sliding Window N-grams]

    E --> G[N-gram Processor<br/>MLP / LSTM / Mamba]
    F --> H[N-gram Processor<br/>MLP / LSTM / Mamba]

    G --> I[Attention Pooling]
    H --> J[Attention Pooling]

    I --> K[Sequence Embedding 1]
    J --> L[Sequence Embedding 2]

    K --> M[Cosine Similarity]
    L --> M

    M --> N[Classifier MLP]
    N --> O[Operon Prediction<br/>Negative / Positive]
```

### Shape of the data flow
The Siamese network shares the same encoder and n-gram processor for both protein sequences. Each branch creates an embedding, the embeddings are compared with cosine similarity, and the classifier outputs the final operon label.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Literal
import math


class SlidingWindowNgrams(nn.Module):
    """
    Create overlapping n-grams from embeddings using sliding window
    """
    def __init__(self, window_size: int = 20, stride: int = 1):
        super().__init__()
        self.window_size = window_size
        self.stride = stride

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings: (batch, seq_len, hidden_dim)

        Returns:
            ngrams: (batch, num_ngrams, hidden_dim)
        """
        batch_size, seq_len, hidden_dim = embeddings.shape

        # Calculate number of n-grams
        num_ngrams = (seq_len - self.window_size) // self.stride + 1

        # Create n-grams by averaging within each window
        ngrams = []
        for i in range(num_ngrams):
            start_idx = i * self.stride
            end_idx = start_idx + self.window_size

            # Average embeddings within this window
            window = embeddings[:, start_idx:end_idx, :]  # (batch, window_size, hidden_dim)
            ngram = window.mean(dim=1)  # (batch, hidden_dim)
            ngrams.append(ngram)

        # Stack all n-grams
        ngrams = torch.stack(ngrams, dim=1)  # (batch, num_ngrams, hidden_dim)

        return ngrams


class AttentionPooling(nn.Module):
    """
    Attention-weighted pooling over n-grams
    Learns which n-grams are important
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.attention = nn.Linear(input_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, num_ngrams, dim)

        Returns:
            pooled: (batch, dim)
        """
        # Compute attention scores
        scores = self.attention(x)  # (batch, num_ngrams, 1)
        weights = F.softmax(scores, dim=1)  # (batch, num_ngrams, 1)

        # Weighted sum
        pooled = (x * weights).sum(dim=1)  # (batch, dim)

        return pooled


class MLPNgramProcessor(nn.Module):
    """
    Process each n-gram independently through MLP
    Architecture A
    """
    def __init__(
        self,
        input_dim: int = 256,
        hidden_dims: list = [256, 128],
        dropout: float = 0.1
    ):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.LayerNorm(hidden_dim)
            ])
            prev_dim = hidden_dim

        self.mlp = nn.Sequential(*layers)
        self.output_dim = prev_dim

    def forward(self, ngrams: torch.Tensor) -> torch.Tensor:
        """
        Args:
            ngrams: (batch, num_ngrams, input_dim)

        Returns:
            output: (batch, num_ngrams, output_dim)
        """
        batch_size, num_ngrams, input_dim = ngrams.shape

        # Reshape to process all n-grams together
        ngrams_flat = ngrams.view(-1, input_dim)  # (batch * num_ngrams, input_dim)

        # Process through MLP
        output_flat = self.mlp(ngrams_flat)  # (batch * num_ngrams, output_dim)

        # Reshape back
        output = output_flat.view(batch_size, num_ngrams, -1)  # (batch, num_ngrams, output_dim)

        return output


class LSTMNgramProcessor(nn.Module):
    """
    Process n-grams as a sequence using LSTM
    Captures relationships between n-grams
    Architecture B
    """
    def __init__(
        self,
        input_dim: int = 256,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.1,
        bidirectional: bool = True
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional,
            batch_first=True
        )

        self.output_dim = hidden_dim * 2 if bidirectional else hidden_dim

    def forward(self, ngrams: torch.Tensor) -> torch.Tensor:
        """
        Args:
            ngrams: (batch, num_ngrams, input_dim)

        Returns:
            output: (batch, num_ngrams, output_dim)
        """
        # Process through LSTM
        output, (h_n, c_n) = self.lstm(ngrams)  # (batch, num_ngrams, output_dim)

        return output


class SiameseNgramProteinNetwork(nn.Module):
    """
    Siamese Network with N-gram processing

    Pipeline:
    1. ESM-2 embeddings → (seq_len, 256)
    2. Sliding window n-grams → (num_ngrams, 256)
    3. Neural network (MLP or LSTM) → (num_ngrams, hidden_dim)
    4. Attention pooling → (hidden_dim,)
    5. Cosine similarity between pair → scalar
    6. Classification → 0 or 1
    """

    def __init__(
        self,
        model_name: str = "facebook/esm2_t12_35M_UR50D",
        ngram_size: int = 20,
        ngram_stride: int = 1,
        processor_type: Literal["mlp", "lstm", "mamba"] = "mlp",
        hidden_dims: list = [256, 128],
        lstm_hidden_dim: int = 128,
        lstm_layers: int = 2,
        mamba_d_state: int = 16,
        mamba_d_conv: int = 4,
        mamba_expand: int = 2,
        mamba_layers: int = 2,
        dropout: float = 0.1,
        freeze_encoder: bool = False
    ):
        """
        Args:
            model_name: ESM-2 model to use
            ngram_size: Size of sliding window (20 or 50)
            ngram_stride: Stride for sliding window (default 1)
            processor_type: "mlp" for Architecture A, "lstm" for Architecture B, "mamba" for Mamba
            hidden_dims: Hidden dimensions for MLP (if processor_type="mlp")
            lstm_hidden_dim: Hidden dimension for LSTM (if processor_type="lstm")
            lstm_layers: Number of LSTM layers
            mamba_d_state: State dimension for Mamba
            mamba_d_conv: Convolution kernel size for Mamba
            mamba_expand: Expansion factor for Mamba
            mamba_layers: Number of Mamba blocks
            dropout: Dropout rate
            freeze_encoder: Freeze ESM-2 encoder
        """
        super().__init__()

        self.ngram_size = ngram_size
        self.processor_type = processor_type

        # Load ESM-2 encoder
        from transformers import EsmModel, EsmTokenizer
        self.encoder = EsmModel.from_pretrained(model_name)
        self.tokenizer = EsmTokenizer.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size  # 256 for ESM-2

        # Freeze encoder if specified
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # N-gram creator
        self.ngram_creator = SlidingWindowNgrams(
            window_size=ngram_size,
            stride=ngram_stride
        )

        # N-gram processor (Architecture A, B, or Mamba)
        if processor_type == "mlp":
            self.ngram_processor = MLPNgramProcessor(
                input_dim=self.hidden_size,
                hidden_dims=hidden_dims,
                dropout=dropout
            )
        elif processor_type == "lstm":
            self.ngram_processor = LSTMNgramProcessor(
                input_dim=self.hidden_size,
                hidden_dim=lstm_hidden_dim,
                num_layers=lstm_layers,
                dropout=dropout,
                bidirectional=True
            )
        elif processor_type == "mamba":
            self.ngram_processor = MambaNgramProcessor(
                input_dim=self.hidden_size,
                d_state=mamba_d_state,
                d_conv=mamba_d_conv,
                expand=mamba_expand,
                num_layers=mamba_layers,
                dropout=dropout
            )
        else:
            raise ValueError(f"Unknown processor_type: {processor_type}")

        # Attention pooling
        self.attention_pooling = AttentionPooling(
            input_dim=self.ngram_processor.output_dim
        )

        # Final classifier (takes cosine similarity as input)
        self.classifier = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

        print(f"\nSiamese N-gram Protein Network:")
        print(f"  Encoder: {model_name}")
        print(f"  N-gram size: {ngram_size}")
        print(f"  Processor: {processor_type.upper()}")
        print(f"  Output dim: {self.ngram_processor.output_dim}")
        print(f"  Frozen encoder: {freeze_encoder}")

    def encode_sequence(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """
        Encode a single protein sequence

        Args:
            input_ids: (batch, seq_len)
            attention_mask: (batch, seq_len)

        Returns:
            final_embedding: (batch, output_dim)
        """
        # Get ESM-2 embeddings
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = outputs.last_hidden_state  # (batch, seq_len, hidden_size)

        # Create n-grams
        ngrams = self.ngram_creator(embeddings)  # (batch, num_ngrams, hidden_size)

        # Process n-grams
        processed_ngrams = self.ngram_processor(ngrams)  # (batch, num_ngrams, output_dim)

        # Attention pooling
        final_embedding = self.attention_pooling(processed_ngrams)  # (batch, output_dim)

        return final_embedding

    def forward(
        self,
        input_ids_1: torch.Tensor,
        input_ids_2: torch.Tensor,
        attention_mask_1: torch.Tensor,
        attention_mask_2: torch.Tensor
    ) -> torch.Tensor:
        """
        Forward pass through Siamese network

        Args:
            input_ids_1: (batch, seq_len)
            input_ids_2: (batch, seq_len)
            attention_mask_1: (batch, seq_len)
            attention_mask_2: (batch, seq_len)

        Returns:
            logits: (batch, 2)
        """
        # Encode both sequences through shared encoder
        emb1 = self.encode_sequence(input_ids_1, attention_mask_1)  # (batch, output_dim)
        emb2 = self.encode_sequence(input_ids_2, attention_mask_2)  # (batch, output_dim)

        # Compute cosine similarity
        cosine_sim = F.cosine_similarity(emb1, emb2, dim=1, eps=1e-8)  # (batch,)
        cosine_sim = cosine_sim.unsqueeze(1)  # (batch, 1)

        # Classify based on similarity
        logits = self.classifier(cosine_sim)  # (batch, 2)

        return logits

    def get_embeddings(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """Get embeddings for analysis"""
        return self.encode_sequence(input_ids, attention_mask)


def create_ngram_siamese_model(
    ngram_size: int = 20,
    processor_type: Literal["mlp", "lstm", "mamba"] = "mlp",
    freeze_encoder: bool = False,
    **kwargs
) -> SiameseNgramProteinNetwork:
    """
    Factory function to create n-gram Siamese model

    Args:
        ngram_size: 20 or 50
        processor_type: "mlp" or "lstm" or "mamba"
        freeze_encoder: Whether to freeze ESM-2

    Returns:
        SiameseNgramProteinNetwork instance
    """
    return SiameseNgramProteinNetwork(
        ngram_size=ngram_size,
        processor_type=processor_type,
        freeze_encoder=freeze_encoder,
        **kwargs
    )

# Mamba-based N-gram Processor

This section introduces the Mamba architecture for processing n-grams within the Siamese network. Mamba is a selective state space model (SSM) that offers efficient sequence modeling with linear scaling, making it a powerful alternative to traditional RNNs or Transformers, especially for long sequences.

### MambaBlock
The core building block of the Mamba architecture, responsible for processing sequences efficiently. It includes components for selective scanning and linear transformations.

### MambaNgramProcessor
This class adapts the `MambaBlock` to process the n-grams generated from the protein sequence embeddings. It can capture dependencies and long-range interactions across the sequence of n-grams, offering a different inductive bias compared to MLP or LSTM processors.

In [10]:
from einops import rearrange, repeat
import torch.nn.functional as F

# Based on `mamba_ssm` library components
# Simplified for demonstration purposes, full implementation might require more details.
class MambaBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        dt_rank: Literal["auto"] | int = "auto",
        dt_min: float = 0.001,
        dt_max: float = 0.1,
        dt_init: str = "random",
        dt_scale: float = 1.0,
        dt_init_floor: float = 1e-4,
        conv_bias: bool = True,
        bias: bool = False,
        use_fast_path: bool = True,
        # For Mamba-2
        initializer_cfg=None,
        res_post_norm: bool = False,
        norm_epsilon: float = 1e-5,
        ngsm=None,
        fused_add_rmsnorm=False,
        fused_dropout_add_rmsnorm=False,
        # For training
        _cuda_graphs=False, # This is usually passed to `mamba_ssm.Mamba`
    ):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.dt_rank = dt_rank if dt_rank != "auto" else math.ceil(self.d_model / 16)

        self.in_proj = nn.Linear(d_model, expand * d_model * 2, bias=bias)

        self.conv1d = nn.Conv1d(
            in_channels=expand * d_model,
            out_channels=expand * d_model,
            kernel_size=d_conv,
            groups=expand * d_model,
            padding=d_conv - 1,
            bias=conv_bias,
        )

        self.x_proj = nn.Linear(expand * d_model, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, expand * d_model, bias=True)

        # S4D real initialization
        A = repeat(torch.arange(1, d_state + 1, dtype=torch.float32), 'n -> d n', d=expand * d_model).contiguous()
        self.A_log = nn.Parameter(torch.log(A)) # A is state matrix
        self.D = nn.Parameter(torch.ones(expand * d_model)) # D is the skip connection

        self.out_proj = nn.Linear(expand * d_model, d_model, bias=bias)

    def forward(self, x):
        # x (batch, seq_len, d_model)
        batch, seq_len, d_model = x.shape

        # 1. Linear projection (in_proj)
        # (batch, seq_len, expand * d_model * 2)
        xz = self.in_proj(x)
        x, z = xz.chunk(2, dim=-1)

        # 2. Conv1d
        # (batch, expand * d_model, seq_len)
        x = rearrange(x, 'b l d -> b d l')
        x = F.silu(self.conv1d(x)[:, :, :seq_len])
        x = rearrange(x, 'b d l -> b l d')

        # 3. State Space Model (SSM) core logic
        # Simplified SSM operation - actual Mamba SSM is more complex and optimized
        # (batch, seq_len, dt_rank + 2 * d_state)
        x_db = self.x_proj(x)
        dt, B, C = x_db.split([self.dt_rank, self.d_state, self.d_state], dim=-1)

        # (batch, seq_len, expand * d_model)
        dt = F.softplus(self.dt_proj(dt))

        A = -torch.exp(self.A_log.float())
        # Simplified recurrence (not true selective scan for brevity)
        # In real mamba_ssm, this is a highly optimized CUDA kernel
        y = torch.zeros(batch, self.expand * d_model, seq_len, device=x.device)
        h = torch.zeros(batch, self.expand * d_model, self.d_state, device=x.device)

        for i in range(seq_len):
            # dt, B, C are (batch, d_inner)
            # x is (batch, d_inner)
            # A is (d_inner, d_state)

            # dt, B, C are (batch, d_inner, d_state)
            _dt = dt[:, i].unsqueeze(-1)
            _B = B[:, i].unsqueeze(1)
            _C = C[:, i].unsqueeze(1)
            _x = x[:, i].unsqueeze(-1)

            # Discrete A, B
            dA = torch.exp(_dt * A)
            dB = _dt * _B

            h = dA * h + dB * _x
            y[:, :, i] = (h @ _C.transpose(-1, -2)).squeeze(-1) + self.D * x[:, i]

        y = rearrange(y, 'b d l -> b l d')

        # 4. Output projection and activation
        y = y * F.silu(z)
        out = self.out_proj(y)

        return out


class MambaNgramProcessor(nn.Module):
    def __init__(
        self,
        input_dim: int = 256,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        num_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.num_layers = num_layers

        # Mamba blocks
        self.layers = nn.ModuleList([
            MambaBlock(d_model=input_dim, d_state=d_state, d_conv=d_conv, expand=expand)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(input_dim) # Added layer norm at the end of stack

        # Output dimension is the same as input_dim for MambaBlock
        self.output_dim = input_dim

    def forward(self, ngrams: torch.Tensor) -> torch.Tensor:
        """
        Args:
            ngrams: (batch, num_ngrams, input_dim)

        Returns:
            output: (batch, num_ngrams, output_dim)
        """
        x = ngrams
        for layer in self.layers:
            x = layer(x) + x # Residual connection
            x = self.dropout(x)

        output = self.norm(x)
        return output

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from tqdm import tqdm
import json
import os
from dataclasses import dataclass
import logging

#from siamese_ngram_architecture import create_ngram_siamese_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


@dataclass
class NgramTrainingConfig:
    """Configuration for training n-gram models"""

    # Model config
    model_name: str = "facebook/esm2_t12_35M_UR50D"
    ngram_size: int = 20  # 20 or 50
    processor_type: str = "mlp"  # "mlp", "lstm", or "mamba"
    freeze_encoder: bool = False

    # MLP config (if processor_type="mlp")
    mlp_hidden_dims: List[int] = None

    # LSTM config (if processor_type="lstm")
    lstm_hidden_dim: int = 128
    lstm_layers: int = 2

    # Mamba config (if processor_type="mamba")
    mamba_d_state: int = 16
    mamba_d_conv: int = 4
    mamba_expand: int = 2
    mamba_layers: int = 2

    # Training config
    batch_size: int = 8  # Smaller due to n-gram processing
    num_epochs: int = 20
    learning_rate: float = 2e-5
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.1
    gradient_clip: float = 1.0
    max_seq_length: int = 512

    # Data config
    train_data_path: str = "operon_train.csv"
    val_data_path: str = "operon_val.csv"

    # Logging
    save_dir: str = "checkpoints_ngram"
    experiment_name: str = "mlp_n20"
    log_interval: int = 50

    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    def __post_init__(self):
        if self.mlp_hidden_dims is None:
            self.mlp_hidden_dims = [256, 128]


class ProteinPairCSVDataset:
    """
    Dataset for protein pairs from CSV
    CSV format: sequence_1, sequence_2, operon_id, label
    """
    def __init__(
        self,
        csv_path: str,
        tokenizer,
        max_length: int = 512
    ):
        """
        Args:
            csv_path: Path to CSV file
            tokenizer: ESM tokenizer
            max_length: Maximum sequence length
        """
        self.max_length = max_length
        self.tokenizer = tokenizer

        # Load CSV
        df = pd.read_csv(csv_path)

        logger.info(f"Pre-tokenizing {len(df)} pairs from {csv_path}...")
        # Tokenize all sequences at once
        self.encodings_1 = self.tokenizer(
            df['sequence_1'].tolist(),
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        self.encodings_2 = self.tokenizer(
            df['sequence_2'].tolist(),
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        self.labels = torch.tensor(df['label'].tolist(), dtype=torch.long)
        self.length = len(df)
        logger.info(f"Finished pre-tokenizing {self.length} pairs.")

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Returns:
            Dictionary with pre-tokenized sequences and label
        """
        return {
            'input_ids_1': self.encodings_1['input_ids'][idx],
            'attention_mask_1': self.encodings_1['attention_mask'][idx],
            'input_ids_2': self.encodings_2['input_ids'][idx],
            'attention_mask_2': self.encodings_2['attention_mask'][idx],
            'label': self.labels[idx]
        }


class NgramTrainer:
    """
    Trainer for N-gram Siamese networks
    """
    def __init__(
        self,
        model,
        config: NgramTrainingConfig,
        train_loader: DataLoader,
        val_loader: DataLoader
    ):
        self.model = model.to(config.device)
        self.config = config
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = config.device

        # Optimizer
        if config.freeze_encoder:
            # Only train n-gram processor and classifier
            trainable_params = [
                {'params': model.ngram_processor.parameters(), 'lr': config.learning_rate},
                {'params': model.attention_pooling.parameters(), 'lr': config.learning_rate},
                {'params': model.classifier.parameters(), 'lr': config.learning_rate}
            ]
        else:
            # Different learning rates for encoder vs rest
            trainable_params = [
                {'params': model.encoder.parameters(), 'lr': config.learning_rate * 0.1},
                {'params': model.ngram_processor.parameters(), 'lr': config.learning_rate},
                {'params': model.attention_pooling.parameters(), 'lr': config.learning_rate},
                {'params': model.classifier.parameters(), 'lr': config.learning_rate}
            ]

        self.optimizer = optim.AdamW(trainable_params, weight_decay=config.weight_decay)

        # Loss
        self.criterion = nn.CrossEntropyLoss()

        # Scheduler
        total_steps = len(train_loader) * config.num_epochs
        warmup_steps = int(total_steps * config.warmup_ratio)

        from transformers import get_linear_schedule_with_warmup
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )

        # AMP Scaler
        self.scaler = torch.cuda.amp.GradScaler()

        # Tracking
        self.global_step = 0
        self.best_val_f1 = 0.0

        # Create save directory
        save_path = os.path.join(config.save_dir, config.experiment_name)
        os.makedirs(save_path, exist_ok=True)
        self.save_path = save_path

        # Save config
        with open(os.path.join(self.save_path, 'config.json'), 'w') as f:
            json.dump(config.__dict__, f, indent=2)

    def train_epoch(self, epoch: int) -> Dict[str, float]:
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0

        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")

        for batch in pbar:
            # Move to device
            input_ids_1 = batch['input_ids_1'].to(self.device)
            input_ids_2 = batch['input_ids_2'].to(self.device)
            attention_mask_1 = batch['attention_mask_1'].to(self.device)
            attention_mask_2 = batch['attention_mask_2'].to(self.device)
            labels = batch['label'].to(self.device)

            self.optimizer.zero_grad()

            # Forward pass with AMP
            with torch.cuda.amp.autocast():
                logits = self.model(
                    input_ids_1, input_ids_2,
                    attention_mask_1, attention_mask_2
                )
                loss = self.criterion(logits, labels)

            # Backward pass with scaler
            self.scaler.scale(loss).backward()

            # Unscale before clipping gradients
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(),
                self.config.gradient_clip
            )

            # Optimizer step & scaler update
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()

            # Track metrics
            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Update progress bar
            current_lr = self.optimizer.param_groups[0]['lr']
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'acc': f"{correct/total:.4f}",
                'lr': f"{current_lr:.2e}"
            })

            self.global_step += 1

        return {
            'loss': total_loss / len(self.train_loader),
            'accuracy': correct / total
        }

    def evaluate(self) -> Dict[str, float]:
        """Evaluate on validation set"""
        self.model.eval()
        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Evaluating", leave=False):
                input_ids_1 = batch['input_ids_1'].to(self.device)
                input_ids_2 = batch['input_ids_2'].to(self.device)
                attention_mask_1 = batch['attention_mask_1'].to(self.device)
                attention_mask_2 = batch['attention_mask_2'].to(self.device)
                labels = batch['label'].to(self.device)

                with torch.cuda.amp.autocast():
                    logits = self.model(
                        input_ids_1, input_ids_2,
                        attention_mask_1, attention_mask_2
                    )

                probs = torch.softmax(logits, dim=-1)
                _, predicted = torch.max(logits, 1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs[:, 1].cpu().numpy())

        # Compute metrics
        accuracy = accuracy_score(all_labels, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='binary'
        )

        try:
            auc = roc_auc_score(all_labels, all_probs)
        except:
            auc = 0.0

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc
        }

    def save_checkpoint(self, filename: str = 'best_model.pt'):
        """Save model checkpoint"""
        filepath = os.path.join(self.save_path, filename)

        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_f1': self.best_val_f1,
            'config': self.config.__dict__
        }, filepath)

        logger.info(f"Checkpoint saved to {filepath}")

    def train(self):
        """Full training loop"""
        logger.info(f"Starting training: {self.config.experiment_name}")
        logger.info(f"Device: {self.config.device}")
        logger.info(f"N-gram size: {self.config.ngram_size}")
        logger.info(f"Processor: {self.config.processor_type}")

        for epoch in range(self.config.num_epochs):
            # Train
            train_metrics = self.train_epoch(epoch)
            logger.info(f"Epoch {epoch} - Train: Loss={train_metrics['loss']:.4f}, Acc={train_metrics['accuracy']:.4f}")

            # Validate
            val_metrics = self.evaluate()
            logger.info(f"Epoch {epoch} - Val: F1={val_metrics['f1']:.4f}, Acc={val_metrics['accuracy']:.4f}, AUC={val_metrics['auc']:.4f}")

            # Save best model
            if val_metrics['f1'] > self.best_val_f1:
                self.best_val_f1 = val_metrics['f1']
                self.save_checkpoint('best_model.pt')
                logger.info(f"✓ New best F1: {self.best_val_f1:.4f}")

        logger.info(f"Training complete! Best F1: {self.best_val_f1:.4f}")

        return {
            'best_f1': self.best_val_f1,
            'final_val_metrics': val_metrics
        }


def run_experiment(
    ngram_size: int,
    processor_type: str,
    train_csv: str,
    val_csv: str,
    freeze_encoder: bool = True
):
    """
    Run a single experiment

    Args:
        ngram_size: 20 or 50
        processor_type: "mlp", "lstm", or "mamba"
        train_csv: Path to training CSV
        val_csv: Path to validation CSV
        freeze_encoder: Whether to freeze ESM-2
    """

    # Create config
    config = NgramTrainingConfig(
        ngram_size=ngram_size,
        processor_type=processor_type,
        train_data_path=train_csv,
        val_data_path=val_csv,
        freeze_encoder=freeze_encoder,
        experiment_name=f"{processor_type}_n{ngram_size}",
        batch_size=8,
        num_epochs=20,
        # Mamba-specific config
        mamba_d_state=16,
        mamba_d_conv=4,
        mamba_expand=2,
        mamba_layers=2
    )

    # Create model
    logger.info(f"\n{'='*80}")
    logger.info(f"EXPERIMENT: {processor_type.upper()} with n={ngram_size}")
    logger.info(f"{'='*80}")

    model = create_ngram_siamese_model(
        ngram_size=ngram_size,
        processor_type=processor_type,
        freeze_encoder=freeze_encoder,
        hidden_dims=config.mlp_hidden_dims if processor_type == "mlp" else None,
        lstm_hidden_dim=config.lstm_hidden_dim if processor_type == "lstm" else None,
        lstm_layers=config.lstm_layers if processor_type == "lstm" else None,
        mamba_d_state=config.mamba_d_state if processor_type == "mamba" else None,
        mamba_d_conv=config.mamba_d_conv if processor_type == "mamba" else None,
        mamba_expand=config.mamba_expand if processor_type == "mamba" else None,
        mamba_layers=config.mamba_layers if processor_type == "mamba" else None
    )

    # Create datasets
    logger.info("Loading datasets...")

    train_dataset = ProteinPairCSVDataset(
        train_csv,
        model.tokenizer,
        max_length=config.max_seq_length
    )

    val_dataset = ProteinPairCSVDataset(
        val_csv,
        model.tokenizer,
        max_length=config.max_seq_length
    )

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # Create trainer
    trainer = NgramTrainer(
        model=model,
        config=config,
        train_loader=train_loader,
        val_loader=val_loader
    )

    # Train
    results = trainer.train()

    return results


def run_all_experiments(train_csv: str, val_csv: str):
    """
    Run all experiments:
    - MLP n=20
    - MLP n=50
    - LSTM n=20
    - LSTM n=50
    - Mamba n=20
    - Mamba n=50
    """

    experiments = [
        {"ngram_size": 20, "processor_type": "mlp"},
        {"ngram_size": 50, "processor_type": "mlp"},
        {"ngram_size": 20, "processor_type": "lstm"},
        {"ngram_size": 50, "processor_type": "lstm"},
        {"ngram_size": 20, "processor_type": "mamba"},
        {"ngram_size": 50, "processor_type": "mamba"}
    ]

    all_results = {}

    for exp in experiments:
        exp_name = f"{exp['processor_type']}_n{exp['ngram_size']}"

        logger.info(f"\n\n{'#'*80}")
        logger.info(f"# RUNNING EXPERIMENT: {exp_name}")
        logger.info(f"{'#'*80}\n")

        results = run_experiment(
            ngram_size=exp['ngram_size'],
            processor_type=exp['processor_type'],
            train_csv=train_csv,
            val_csv=val_csv,
            freeze_encoder=True
        )

        all_results[exp_name] = results

    # Save summary
    summary_df = pd.DataFrame([
        {
            'experiment': name,
            'best_f1': res['best_f1'],
            'final_accuracy': res['final_val_metrics']['accuracy'],
            'final_auc': res['final_val_metrics']['auc']
        }
        for name, res in all_results.items()
    ])

    summary_df.to_csv('checkpoints_ngram/experiment_summary.csv', index=False)

    logger.info("\n\n" + "="*80)
    logger.info("ALL EXPERIMENTS COMPLETE!")
    logger.info("="*80)
    logger.info("\nSummary:")
    print(summary_df.to_string(index=False))

    return all_results

In [12]:
import pickle
import os
import logging # Ensure logging is imported here

logger = logging.getLogger(__name__)

def _execute_and_save_single_experiment_results(ngram_size: int, processor_type: str):
    """
    Sets up paths, runs a single experiment, and saves its results.
    """
    # Paths to CSV files
    train_csv = TRAIN_CSV
    val_csv = VAL_CSV

    # Check if files exist
    if not os.path.exists(train_csv):
        logger.error(f"Training CSV not found: {train_csv}")
        logger.error("Please ensure 'operon_train.csv' is in your Drive.")
        return

    if not os.path.exists(val_csv):
        logger.error(f"Validation CSV not found: {val_csv}")
        logger.error("Please ensure 'operon_val.csv' is in your Drive.")
        return

    exp_name = f"{processor_type}_n{ngram_size}"
    logger.info(f"\n\n{'#'*80}")
    logger.info(f"# RUNNING INDIVIDUAL EXPERIMENT: {exp_name}")
    logger.info(f"{'#'*80}\n")

    # Run the experiment
    results = run_experiment(
        ngram_size=ngram_size,
        processor_type=processor_type,
        train_csv=train_csv,
        val_csv=val_csv,
        freeze_encoder=True # Assuming freeze_encoder is consistent for these runs
    )

    # Save the individual experiment results to a pickle file in Google Drive
    output_dir = OPERON_DATA_DIR
    os.makedirs(output_dir, exist_ok=True)
    results_filepath = os.path.join(output_dir, f"{exp_name}_results.pkl")
    with open(results_filepath, 'wb') as f:
        pickle.dump(results, f)
    logger.info(f"Results for {exp_name} saved to {results_filepath}")

    logger.info(f"\nExperiment {exp_name} complete! Check {output_dir} for results.")
    print(f"Results for {exp_name}: {results}") # Print results for immediate feedback

In [13]:
# Run the first experiment: MLP with n-gram size 20
_execute_and_save_single_experiment_results(20, "mlp")

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]


Siamese N-gram Protein Network:
  Encoder: facebook/esm2_t12_35M_UR50D
  N-gram size: 20
  Processor: MLP
  Output dim: 128
  Frozen encoder: True


Epoch 19: 100%|██████████| 4641/4641 [02:31<00:00, 30.61it/s, loss=0.1860, acc=0.8764, lr=0.00e+00]
                                                               

Results for mlp_n20: {'best_f1': 0.7772972253642957, 'final_val_metrics': {'accuracy': 0.7896556058320764, 'precision': 0.8288288288288288, 'recall': 0.7294236093632016, 'f1': 0.775955552580494, 'auc': np.float64(0.8640711271709194)}}


In [ ]:
# Run the second experiment: MLP with n-gram size 50
#_execute_and_save_single_experiment_results(50, "mlp")

In [15]:
# Run the third experiment: LSTM with n-gram size 20
_execute_and_save_single_experiment_results(20, "lstm")

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Siamese N-gram Protein Network:
  Encoder: facebook/esm2_t12_35M_UR50D
  N-gram size: 20
  Processor: LSTM
  Output dim: 256
  Frozen encoder: True


Epoch 19: 100%|██████████| 4641/4641 [03:33<00:00, 21.75it/s, loss=0.5238, acc=0.7913, lr=0.00e+00]


Results for lstm_n20: {'best_f1': 0.7482023231516195, 'final_val_metrics': {'accuracy': 0.7447209653092006, 'precision': 0.7408234126984127, 'recall': 0.7518248175182481, 'f1': 0.7462835727670206, 'auc': np.float64(0.8209409797200651)}}


In [ ]:
# Run the fourth experiment: LSTM with n-gram size 50
#_execute_and_save_single_experiment_results(50, "lstm")

In [ ]:
# Run the fifth experiment: Mamba with n-gram size 20
_execute_and_save_single_experiment_results(20, "mamba")

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Siamese N-gram Protein Network:
  Encoder: facebook/esm2_t12_35M_UR50D
  N-gram size: 20
  Processor: MAMBA
  Output dim: 480
  Frozen encoder: True


Epoch 2:  29%|██▉       | 1350/4641 [16:44<40:12,  1.36it/s, loss=0.5729, acc=0.6412, lr=1.97e-05]

In [ ]:
# Run the sixth experiment: Mamba with n-gram size 50
#_execute_and_save_single_experiment_results(50, "mamba")